<a href="https://colab.research.google.com/github/Mohd-Abdul-Muqeet/FlyRank-AI/blob/main/work/notebooks/w04_signal_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Mohd-Abdul-Muqeet/FlyRank-AI-Week1/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

In [3]:
import os
import numpy as np
import pandas as pd
from datasets import load_dataset

# Load the gated warehouse.
# Authentication must already be active in Colab.
ds = load_dataset(
    "FlyRank/internship-warehouse",
    "fact_content_daily_performance",
    streaming=True,
    split="train"
)

# Reproducible working sample.
df = pd.DataFrame(list(ds.take(10000)))

# Numeric fields used in this audit.
numeric_cols = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "sessions_organic",
    "sessions_direct",
    "sessions_referral",
    "sessions_social",
    "sessions_paid",
    "sessions_ai",
]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)

# Safe observed CTR calculation.
df["CTR"] = np.where(
    df["gsc_impressions"] > 0,
    df["gsc_clicks"] / df["gsc_impressions"],
    np.nan
)

print("Rows:", len(df))
print("Columns:", len(df.columns))

print("\nKey field distributions:")
print(
    df[
        [
            "gsc_impressions",
            "gsc_clicks",
            "gsc_avg_position",
            "CTR",
            "sessions_organic",
            "sessions_ai",
        ]
    ].describe(
        percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
    ).T
)

print("\nMissing values:")
print(df[numeric_cols].isna().sum())

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

Rows: 10000
Columns: 31

Key field distributions:
                    count       mean        std  min  25%    50%   75%  \
gsc_impressions   10000.0  11.580600  19.356807  1.0  2.0   6.00  14.0   
gsc_clicks        10000.0   0.105900   0.430701  0.0  0.0   0.00   0.0   
gsc_avg_position  10000.0  24.988312  22.666650  0.0  8.0  15.75  36.5   
CTR               10000.0   0.010478   0.061232  0.0  0.0   0.00   0.0   
sessions_organic  10000.0   0.000000   0.000000  0.0  0.0   0.00   0.0   
sessions_ai       10000.0   0.000000   0.000000  0.0  0.0   0.00   0.0   

                        90%        95%    99%    max  
gsc_impressions   27.000000  38.000000  88.00  506.0  
gsc_clicks         0.000000   1.000000   2.00   11.0  
gsc_avg_position  60.803333  75.293048  92.00  127.0  
CTR                0.000000   0.055556   0.25    1.0  
sessions_organic   0.000000   0.000000   0.00    0.0  
sessions_ai        0.000000   0.000000   0.00    0.0  

Missing values:
gsc_impressions      0
gsc_cl

## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

In [4]:
# Signal test #1:
# Does better observed search position correspond to higher CTR?

position_df = df[
    (df["gsc_impressions"] > 0) &
    (df["gsc_avg_position"] > 0) &
    (df["CTR"].notna())
].copy()

position_df["position_tier"] = pd.cut(
    position_df["gsc_avg_position"],
    bins=[0, 3, 10, 20, 50, np.inf],
    labels=["1-3", "4-10", "11-20", "21-50", "50+"],
    include_lowest=True
)

position_result = (
    position_df
    .groupby("position_tier", observed=False)
    .agg(
        rows=("CTR", "size"),
        mean_ctr=("CTR", "mean"),
        median_ctr=("CTR", "median"),
        mean_impressions=("gsc_impressions", "mean")
    )
    .reset_index()
)

print("SIGNAL 1 — Search position vs CTR")
print(position_result.to_string(index=False))

# Signal test #2:
# Does search exposure affect how reliable/useful CTR is?

exposure_df = df[
    (df["gsc_impressions"] > 0) &
    (df["CTR"].notna())
].copy()

exposure_df["exposure_tier"] = pd.qcut(
    exposure_df["gsc_impressions"].rank(method="first"),
    q=4,
    labels=["Q1-low", "Q2", "Q3", "Q4-high"]
)

exposure_result = (
    exposure_df
    .groupby("exposure_tier", observed=False)
    .agg(
        rows=("CTR", "size"),
        mean_impressions=("gsc_impressions", "mean"),
        mean_ctr=("CTR", "mean"),
        median_ctr=("CTR", "median")
    )
    .reset_index()
)

print("\nSIGNAL 2 — Exposure vs CTR")
print(exposure_result.to_string(index=False))

# Signal test #3:
# Is organic traffic associated with observed search performance?

organic_df = df.copy()
organic_df["has_organic"] = (
    organic_df["sessions_organic"] > 0
)

organic_result = (
    organic_df
    .groupby("has_organic")
    .agg(
        rows=("CTR", "size"),
        mean_organic_sessions=("sessions_organic", "mean"),
        mean_ctr=("CTR", "mean"),
        median_ctr=("CTR", "median")
    )
    .reset_index()
)

print("\nSIGNAL 3 — Organic sessions vs CTR")
print(organic_result.to_string(index=False))

print("\nVerdict rule:")
print("CONFIRMED = direction is consistently visible across groups.")
print("MIXED = direction is not consistent.")
print("OPPOSITE = observed direction contradicts the proposed signal.")
print("FALSE = little/no observable relationship in this sample.")

SIGNAL 1 — Search position vs CTR
position_tier  rows  mean_ctr  median_ctr  mean_impressions
          1-3   297  0.036162         0.0         21.996633
         4-10  3482  0.014690         0.0         11.040781
        11-20  1832  0.011343         0.0         12.523472
        21-50  2874  0.005468         0.0         11.949200
          50+  1484  0.000942         0.0          9.068059

SIGNAL 2 — Exposure vs CTR
exposure_tier  rows  mean_impressions  mean_ctr  median_ctr
       Q1-low  2500            1.3232  0.011000         0.0
           Q2  2500            3.8820  0.011513         0.0
           Q3  2500            9.2840  0.010368         0.0
      Q4-high  2500           31.8332  0.009032         0.0

SIGNAL 3 — Organic sessions vs CTR
 has_organic  rows  mean_organic_sessions  mean_ctr  median_ctr
       False 10000                    0.0  0.010478         0.0

Verdict rule:
CONFIRMED = direction is consistently visible across groups.
MIXED = direction is not consistent.
O

## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

In [5]:
# Flag-linked test:
# Test the assumption that a page with meaningful search exposure
# and weak CTR can be a useful review candidate.

flag_df = df[
    (df["gsc_impressions"] >= 10) &
    (df["gsc_avg_position"] > 0) &
    (df["CTR"].notna())
].copy()

flag_df["visible"] = (
    flag_df["gsc_avg_position"] <= 20
)

flag_df["low_ctr"] = (
    flag_df["CTR"] < 0.02
)

flag_result = (
    flag_df
    .groupby(["visible", "low_ctr"])
    .agg(
        rows=("CTR", "size"),
        mean_impressions=("gsc_impressions", "mean"),
        mean_position=("gsc_avg_position", "mean"),
        mean_ctr=("CTR", "mean")
    )
    .reset_index()
)

print("FLAG-LINKED TEST")
print("=" * 70)
print(flag_result.to_string(index=False))

visible_low_ctr = flag_df[
    flag_df["visible"] & flag_df["low_ctr"]
]

print("\nVisible + low-CTR rows:",
      len(visible_low_ctr))

print(
    "\nObserved interpretation: pages with meaningful exposure and "
    "weak observed CTR can be treated as review candidates. "
    "This is directional decision-support evidence, not a causal claim."
)

FLAG-LINKED TEST
 visible  low_ctr  rows  mean_impressions  mean_position  mean_ctr
   False    False    79         24.506329      30.127605  0.055850
   False     True  1494         23.753681      41.130926  0.000069
    True    False   411         30.270073       8.428912  0.064902
    True     True  1583         26.561592      10.074934  0.000474

Visible + low-CTR rows: 1583

Observed interpretation: pages with meaningful exposure and weak observed CTR can be treated as review candidates. This is directional decision-support evidence, not a causal claim.


## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

In [6]:
print("PRACTICAL TAKEAWAYS")
print("=" * 70)

print(
    "1. Search exposure should be considered when interpreting CTR, "
    "because very low exposure can make the observed rate unstable."
)

print(
    "2. Average position is useful as a directional visibility signal, "
    "but the audit does not establish that position causes CTR."
)

print(
    "3. Pages with meaningful exposure and relatively weak CTR can be "
    "prioritised for human review rather than automatically changed."
)

print(
    "These findings are observational and should be treated as "
    "decision-support signals."
)

PRACTICAL TAKEAWAYS
1. Search exposure should be considered when interpreting CTR, because very low exposure can make the observed rate unstable.
2. Average position is useful as a directional visibility signal, but the audit does not establish that position causes CTR.
3. Pages with meaningful exposure and relatively weak CTR can be prioritised for human review rather than automatically changed.
These findings are observational and should be treated as decision-support signals.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.